In [2]:
!pip -q install pyspark

In [3]:
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
spark = SparkSession.builder.appName("StreamingSet4").getOrCreate()
files.upload()
def read_csv(name):
  return spark.read.csv(name, header=True, inferSchema=True)
sessions_df = read_csv("watch_sessions_80000.csv")
extra_df = read_csv("watch_sessions_extra_10000.csv")
subscribers_df = read_csv("subscriber_master_2400.csv")
content_df = read_csv("content_catalog_120.csv")
devices_df = read_csv("device_master_8.csv")
plans_df = read_csv("subscription_plans_4.csv")
print("Main sessions:", sessions_df.count())
sessions_df.printSchema()
sessions_df.show(5, truncate=False)

Saving content_catalog_120.csv to content_catalog_120.csv
Saving device_master_8.csv to device_master_8.csv
Saving subscriber_master_2400.csv to subscriber_master_2400.csv
Saving subscription_plans_4.csv to subscription_plans_4.csv
Saving watch_sessions_80000.csv to watch_sessions_80000.csv
Saving watch_sessions_extra_10000.csv to watch_sessions_extra_10000.csv
Main sessions: 80000
root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

+----------+-------------+----------+---------+-------------------+----

**Part A**

In [4]:
sessions_df.printSchema()

print("Total rows:", sessions_df.count())

sessions_df.show(10, truncate=False)

numeric_fields = [
    field.name for field in sessions_df.schema.fields
    if field.dataType.typeName() in
    ["integer", "long", "float", "double", "short", "decimal"]
]

string_fields = [
    field.name for field in sessions_df.schema.fields
    if field.dataType.typeName() == "string"
]

timestamp_fields = [
    field.name for field in sessions_df.schema.fields
    if field.dataType.typeName() == "timestamp"
]

print("Numeric fields:", numeric_fields)
print("String fields:", string_fields)
print("Timestamp fields:", timestamp_fields)

root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

Total rows: 80000
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+--

In [5]:
selected_sessions = sessions_df.select(
    "session_id",
    "subscriber_id",
    "content_id",
    "watch_minutes",
    "session_status"
)

completed_over_60 = selected_sessions.filter(
    (F.col("session_status") == "Completed") &
    (F.col("watch_minutes") > 60)
)

print("Matching sessions:", completed_over_60.count())

completed_over_60.show(10, truncate=False)

Matching sessions: 3403
+----------+-------------+----------+-------------+--------------+
|session_id|subscriber_id|content_id|watch_minutes|session_status|
+----------+-------------+----------+-------------+--------------+
|WS00000049|SUB01859     |CT0075    |125          |Completed     |
|WS00000179|SUB00935     |CT0054    |62           |Completed     |
|WS00000184|SUB00072     |CT0119    |91           |Completed     |
|WS00000222|SUB01603     |CT0103    |116          |Completed     |
|WS00000233|SUB01233     |CT0027    |158          |Completed     |
|WS00000244|SUB00985     |CT0048    |140          |Completed     |
|WS00000249|SUB00681     |CT0100    |120          |Completed     |
|WS00000257|SUB02115     |CT0120    |68           |Completed     |
|WS00000292|SUB01068     |CT0048    |141          |Completed     |
|WS00000326|SUB00368     |CT0096    |97           |Completed     |
+----------+-------------+----------+-------------+--------------+
only showing top 10 rows


In [6]:
engagement_df = sessions_df.withColumn(
    "watch_hours",
    F.col("watch_minutes") / 60.0
).withColumn(
    "high_engagement",
    F.when(
        F.col("completion_pct").isNull(),
        F.lit(None).cast("boolean")
    ).otherwise(
        F.col("completion_pct") >= 75
    )
)

engagement_df.select(
    "session_id",
    "watch_minutes",
    "watch_hours",
    "completion_pct",
    "high_engagement"
).show(10, truncate=False)

+----------+-------------+-------------------+--------------+---------------+
|session_id|watch_minutes|watch_hours        |completion_pct|high_engagement|
+----------+-------------+-------------------+--------------+---------------+
|WS00000001|6            |0.1                |11.8          |false          |
|WS00000002|31           |0.5166666666666667 |56.4          |false          |
|WS00000003|29           |0.48333333333333334|63.0          |false          |
|WS00000004|69           |1.15               |75.0          |true           |
|WS00000005|36           |0.6                |23.4          |false          |
|WS00000006|20           |0.3333333333333333 |19.8          |false          |
|WS00000007|78           |1.3                |83.9          |true           |
|WS00000008|22           |0.36666666666666664|66.7          |false          |
|WS00000009|21           |0.35               |80.8          |true           |
|WS00000010|21           |0.35               |65.6          |fal

In [7]:
top_20_sessions = sessions_df.orderBy(
    F.col("data_used_mb").desc_nulls_last()
)

top_20_sessions.show(20, truncate=False)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00051169|SUB00155     |CT0069    |DV08     |2026-07-20 00:11:00|167          |97.7          |9.0              |4K            |Completed     |8881.8      |3.0        |
|WS00074513|SUB00684     |CT0056    |DV05     |2026-08-09 21:53:00|157          |96.9          |8.0              |4K            |Completed     |8408.4      |NULL       |
|WS00039201|SUB01664     |CT0027    |DV08     |2026-05-16 06:35:00|161          |97.0          |9.0              |4K            |Completed     |8282.0

**Part B**

In [8]:
null_columns = [
    "subscriber_id",
    "device_id",
    "completion_pct",
    "buffering_seconds",
    "user_rating"
]

sessions_df.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c + "_missing")
    for c in null_columns
]).show(truncate=False)

+---------------------+-----------------+----------------------+-------------------------+-------------------+
|subscriber_id_missing|device_id_missing|completion_pct_missing|buffering_seconds_missing|user_rating_missing|
+---------------------+-----------------+----------------------+-------------------------+-------------------+
|951                  |2003             |3513                  |4424                     |77605              |
+---------------------+-----------------+----------------------+-------------------------+-------------------+



In [9]:
buffering_filled_df = sessions_df.withColumn(
    "buffering_seconds",
    F.coalesce(
        F.col("buffering_seconds"),
        F.lit(0)
    )
)

print("Original missing buffering values:",
      sessions_df.filter(
          F.col("buffering_seconds").isNull()
      ).count())

print("New missing buffering values:",
      buffering_filled_df.filter(
          F.col("buffering_seconds").isNull()
      ).count())

buffering_filled_df.show(10, truncate=False)

Original missing buffering values: 4424
New missing buffering values: 0
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00000001|SUB00157     |CT0097    |DV02     |2026-04-26 18:59:00|6            |11.8          |29.0             |HD            |Abandoned     |72.4        |NULL       |
|WS00000002|SUB01430     |CT0031    |DV04     |2026-07-30 06:26:00|31           |56.4          |31.0             |HD            |Interrupted   |422.7       |NULL       |
|WS00000003|SUB01892     |CT0044    |DV08     |2026-02-22 20:55:00|29         

In [10]:
sessions_with_device_df = sessions_df.filter(
    F.col("device_id").isNotNull()
)

original_count = sessions_df.count()
filtered_count = sessions_with_device_df.count()

print("Original row count:", original_count)
print("Rows with device_id:", filtered_count)
print("Rows excluded:", original_count - filtered_count)

sessions_with_device_df.show(10, truncate=False)

Original row count: 80000
Rows with device_id: 77997
Rows excluded: 2003
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00000001|SUB00157     |CT0097    |DV02     |2026-04-26 18:59:00|6            |11.8          |29.0             |HD            |Abandoned     |72.4        |NULL       |
|WS00000002|SUB01430     |CT0031    |DV04     |2026-07-30 06:26:00|31           |56.4          |31.0             |HD            |Interrupted   |422.7       |NULL       |
|WS00000003|SUB01892     |CT0044    |DV08     |2026-02-22 20:55:00|29        

In [11]:
completion_category_df = sessions_df.withColumn(
    "completion_category",
    F.when(
        F.col("completion_pct").isNull(), "Unknown"
    ).when(
        F.col("completion_pct") < 30, "Low"
    ).when(
        F.col("completion_pct") < 90, "Medium"
    ).otherwise(
        "High"
    )
)

completion_category_df.select(
    "session_id",
    "completion_pct",
    "completion_category"
).show(20, truncate=False)

completion_category_df.groupBy(
    "completion_category"
).count().orderBy(
    "completion_category"
).show()

+----------+--------------+-------------------+
|session_id|completion_pct|completion_category|
+----------+--------------+-------------------+
|WS00000001|11.8          |Low                |
|WS00000002|56.4          |Medium             |
|WS00000003|63.0          |Medium             |
|WS00000004|75.0          |Medium             |
|WS00000005|23.4          |Low                |
|WS00000006|19.8          |Low                |
|WS00000007|83.9          |Medium             |
|WS00000008|66.7          |Medium             |
|WS00000009|80.8          |Medium             |
|WS00000010|65.6          |Medium             |
|WS00000011|36.1          |Medium             |
|WS00000012|78.6          |Medium             |
|WS00000013|51.6          |Medium             |
|WS00000014|28.1          |Low                |
|WS00000015|16.4          |Low                |
|WS00000016|28.0          |Low                |
|WS00000017|84.3          |Medium             |
|WS00000018|83.3          |Medium       

**Part C**

In [12]:
genre_metrics = (
    sessions_df
    .join(content_df, "content_id", "left")
    .groupBy("genre")
    .agg(
        F.count("*").alias("session_count"),
        F.sum("watch_minutes").alias("total_watch_minutes")
    )
    .orderBy(F.col("session_count").desc())
)

genre_metrics.show(truncate=False)

+-----------+-------------+-------------------+
|genre      |session_count|total_watch_minutes|
+-----------+-------------+-------------------+
|Adventure  |15561        |839700             |
|Animation  |11886        |490335             |
|Drama      |10733        |519872             |
|Sports     |9967         |383044             |
|Sci-Fi     |9344         |433946             |
|Comedy     |9105         |452563             |
|Documentary|7432         |256626             |
|Thriller   |5972         |248430             |
+-----------+-------------+-------------------+



In [13]:
device_metrics = (
    sessions_df
    .join(devices_df, "device_id", "left")
    .groupBy("device_type")
    .agg(
        F.avg("buffering_seconds").alias("avg_buffering_seconds"),
        F.avg("watch_minutes").alias("avg_watch_minutes"),
        F.count("*").alias("session_count")
    )
    .orderBy("device_type")
)

device_metrics.show(truncate=False)

+---------------+---------------------+------------------+-------------+
|device_type    |avg_buffering_seconds|avg_watch_minutes |session_count|
+---------------+---------------------+------------------+-------------+
|NULL           |35.57619248628113    |44.7866344605475  |2484         |
|Android Phone  |35.676486693680864   |45.0885616509385  |9643         |
|Desktop Browser|36.02889125799574    |45.571818733562615|9886         |
|Game Console   |35.51243726816496    |45.04469158686026 |9711         |
|Laptop Browser |36.930938301907844   |45.84465713083719 |9508         |
|Smart TV       |36.67392739273927    |45.33776623376623 |9625         |
|Streaming Stick|36.320860309448044   |45.08570541351827 |9661         |
|Tablet         |35.23614616481949    |45.38943894389439 |9696         |
|iPhone         |35.761106907361366   |45.226752503576535|9786         |
+---------------+---------------------+------------------+-------------+



In [14]:
session_subscribers = (
    sessions_df
    .filter(F.col("subscriber_id").isNotNull())
    .join(
        subscribers_df.select("subscriber_id", "plan_id"),
        "subscriber_id",
        "left"
    )
)

plan_sessions = session_subscribers.join(
    plans_df.select("plan_id", "plan_name"),
    "plan_id",
    "left"
)

completion_by_plan = (
    plan_sessions
    .groupBy("plan_name")
    .agg(
        F.count("*").alias("all_sessions"),
        F.sum(
            F.when(F.col("session_status") == "Completed", 1)
             .otherwise(0)
        ).alias("completed_sessions")
    )
    .withColumn(
        "completion_rate",
        F.col("completed_sessions") / F.col("all_sessions")
    )
)

completion_by_plan.show(truncate=False)

+---------+------------+------------------+-------------------+
|plan_name|all_sessions|completed_sessions|completion_rate    |
+---------+------------+------------------+-------------------+
|Premium  |12198       |975               |0.07993113625184456|
|NULL     |571         |42                |0.07355516637478109|
|Basic    |25086       |2006              |0.07996492067288527|
|Mobile   |17510       |1429              |0.08161050828098229|
|Standard |23684       |1868              |0.07887181219388617|
+---------+------------+------------------+-------------------+



In [15]:
active_subscribers = (
    subscribers_df
    .filter(F.col("subscriber_status") == "Active")
    .select("subscriber_id", "plan_id")
    .dropDuplicates(["subscriber_id"])
)

mrr_by_plan = (
    active_subscribers
    .join(plans_df, "plan_id", "left")
    .groupBy("plan_name")
    .agg(
        F.countDistinct("subscriber_id").alias("active_subscribers"),
        F.sum("monthly_fee_inr").alias("estimated_monthly_revenue_inr")
    )
    .orderBy("plan_name")
)

mrr_by_plan.show(truncate=False)

total_mrr = mrr_by_plan.agg(
    F.sum("estimated_monthly_revenue_inr").alias("total_mrr")
)

total_mrr.show()

+---------+------------------+-----------------------------+
|plan_name|active_subscribers|estimated_monthly_revenue_inr|
+---------+------------------+-----------------------------+
|Basic    |632               |157368                       |
|Mobile   |443               |66007                        |
|Premium  |326               |211574                       |
|Standard |604               |240996                       |
+---------+------------------+-----------------------------+

+---------+
|total_mrr|
+---------+
|   675945|
+---------+



In [16]:
known_subscribers = subscribers_df.select(
    "subscriber_id"
).dropDuplicates()

known_devices = devices_df.select(
    "device_id"
).dropDuplicates()

unmatched_subscribers = (
    sessions_df
    .filter(F.col("subscriber_id").isNotNull())
    .join(
        known_subscribers,
        "subscriber_id",
        "left_anti"
    )
)

unmatched_devices = (
    sessions_df
    .filter(F.col("device_id").isNotNull())
    .join(
        known_devices,
        "device_id",
        "left_anti"
    )
)

print(
    "Unmatched subscriber sessions:",
    unmatched_subscribers.count()
)

print(
    "Unmatched device sessions:",
    unmatched_devices.count()
)

print("Sample unmatched subscriber records:")
unmatched_subscribers.show(10, truncate=False)

print("Sample unmatched device records:")
unmatched_devices.show(10, truncate=False)

Unmatched subscriber sessions: 571
Unmatched device sessions: 481
Sample unmatched subscriber records:
+-------------+----------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|subscriber_id|session_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+-------------+----------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|SUB99999     |WS00000325|CT0092    |DV02     |2026-01-08 17:10:00|35           |57.4          |17.0             |HD            |Interrupted   |404.2       |NULL       |
|SUB99999     |WS00000353|CT0085    |DV03     |2026-07-25 22:03:00|31           |37.3          |26.0             |HD            |Interrupted   |374.8       |NULL       |
|SUB99999     |WS00000525|CT0064    |DV08     |

**Part D**

In [18]:
sessions_df = sessions_df.withColumn(
    "session_start",
    F.to_timestamp("session_start")
)

monthly_trends = (
    sessions_df
    .filter(F.year("session_start") == 2026)
    .groupBy(F.date_format("session_start", "yyyy-MM").alias("month"))
    .agg(
        F.count("*").alias("total_sessions"),
        F.sum("watch_minutes").alias("total_watch_minutes")
    )
    .orderBy("month")
)

monthly_trends.show(truncate=False)

+-------+--------------+-------------------+
|month  |total_sessions|total_watch_minutes|
+-------+--------------+-------------------+
|2026-01|9070          |407215             |
|2026-02|8218          |372726             |
|2026-03|9019          |403971             |
|2026-04|8817          |399432             |
|2026-05|9006          |407386             |
|2026-06|8776          |404203             |
|2026-07|9204          |416104             |
|2026-08|8996          |407374             |
|2026-09|8894          |406105             |
+-------+--------------+-------------------+



In [19]:
subscribers_df = subscribers_df.withColumn(
    "signup_date",
    F.to_date("signup_date")
)

subscriber_sessions = (
    sessions_df
    .filter(F.col("subscriber_id").isNotNull())
    .join(
        subscribers_df.select("subscriber_id", "signup_date"),
        on="subscriber_id",
        how="inner"
    )
    .withColumn(
        "days_to_session",
        F.datediff(
            F.to_date("session_start"),
            F.col("signup_date")
        )
    )
)

subscriber_sessions.select(
    "subscriber_id",
    "session_id",
    "signup_date",
    "session_start",
    "days_to_session"
).orderBy("session_start").show(10, truncate=False)

+-------------+----------+-----------+-------------------+---------------+
|subscriber_id|session_id|signup_date|session_start      |days_to_session|
+-------------+----------+-----------+-------------------+---------------+
|SUB01707     |WS00035235|2024-02-13 |2026-01-01 00:00:00|688            |
|SUB02206     |WS00074739|2025-08-31 |2026-01-01 00:04:00|123            |
|SUB01248     |WS00032058|2024-03-17 |2026-01-01 00:04:00|655            |
|SUB02342     |WS00048560|2025-08-25 |2026-01-01 00:14:00|129            |
|SUB00563     |WS00067087|2024-08-11 |2026-01-01 00:15:00|508            |
|SUB00711     |WS00004245|2024-10-26 |2026-01-01 00:18:00|432            |
|SUB00261     |WS00041710|2025-05-15 |2026-01-01 00:21:00|231            |
|SUB02167     |WS00045132|2025-11-22 |2026-01-01 00:28:00|40             |
|SUB01237     |WS00036438|2024-02-01 |2026-01-01 00:41:00|700            |
|SUB00646     |WS00030050|2024-11-20 |2026-01-01 00:51:00|407            |
+-------------+----------

In [20]:
sessions_with_day = sessions_df.withColumn(
    "day_of_week",
    F.dayofweek("session_start")
)

sessions_with_day = sessions_with_day.withColumn(
    "day_type",
    F.when(
        F.col("day_of_week").isin(1, 7),
        "Weekend"
    ).otherwise("Weekday")
)

viewing_comparison = (
    sessions_with_day
    .groupBy("day_type")
    .agg(
        F.count("*").alias("total_sessions"),
        F.avg("watch_minutes").alias("average_watch_minutes")
    )
)

viewing_comparison.show(truncate=False)

+--------+--------------+---------------------+
|day_type|total_sessions|average_watch_minutes|
+--------+--------------+---------------------+
|Weekday |57228         |45.26460823373174    |
|Weekend |22772         |45.411601967328295   |
+--------+--------------+---------------------+



In [21]:
genre_content = (
    sessions_df
    .join(
        content_df.select("content_id", "content_title", "genre"),
        on="content_id",
        how="inner"
    )
    .groupBy("genre", "content_id", "content_title")
    .agg(
        F.sum("watch_minutes").alias("total_watch_minutes")
    )
)

genre_window = Window.partitionBy("genre").orderBy(
    F.desc("total_watch_minutes")
)

top_content = (
    genre_content
    .withColumn("genre_rank", F.dense_rank().over(genre_window))
    .filter(F.col("genre_rank") <= 3)
    .orderBy("genre", "genre_rank")
)

top_content.show(truncate=False)

+-----------+----------+----------------+-------------------+----------+
|genre      |content_id|content_title   |total_watch_minutes|genre_rank|
+-----------+----------+----------------+-------------------+----------+
|Adventure  |CT0105    |Infinite Horizon|66395              |1         |
|Adventure  |CT0048    |Lost Journey    |61823              |2         |
|Adventure  |CT0042    |Solar Memory    |60540              |3         |
|Animation  |CT0106    |Last River      |51398              |1         |
|Animation  |CT0114    |Emerald Circuit |51310              |2         |
|Animation  |CT0100    |Golden Puzzle   |50515              |3         |
|Comedy     |CT0027    |Blue Horizon    |61428              |1         |
|Comedy     |CT0019    |Bright Valley   |60921              |2         |
|Comedy     |CT0059    |Silver Valley   |60422              |3         |
|Documentary|CT0077    |Fading Signal   |47085              |1         |
|Documentary|CT0057    |Emerald Valley  |40819     

In [22]:
subscriber_window = Window.partitionBy("subscriber_id").orderBy(
    F.desc("session_start"),
    F.desc("session_id")
)

latest_sessions = (
    sessions_df
    .filter(F.col("subscriber_id").isNotNull())
    .withColumn(
        "session_rank",
        F.row_number().over(subscriber_window)
    )
    .filter(F.col("session_rank") == 1)
    .drop("session_rank")
)

latest_sessions.show(10, truncate=False)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00020822|SUB00001     |CT0073    |DV03     |2026-09-11 19:19:00|34           |89.5          |145.0            |HD            |Interrupted   |418.3       |NULL       |
|WS00032744|SUB00002     |CT0076    |DV05     |2026-09-02 05:41:00|45           |73.8          |219.0            |Full HD       |Interrupted   |1184.1      |NULL       |
|WS00006110|SUB00003     |CT0004    |DV03     |2026-09-30 03:13:00|11           |15.3          |3.0              |4K            |Abandoned     |591.1 

In [23]:
watch_window = Window.partitionBy("subscriber_id").orderBy(
    "session_start",
    "session_id"
)

watch_time_change = (
    sessions_df
    .filter(F.col("subscriber_id").isNotNull())
    .withColumn(
        "previous_watch_minutes",
        F.lag("watch_minutes", 1).over(watch_window)
    )
    .withColumn(
        "watch_minutes_change",
        F.col("watch_minutes") - F.col("previous_watch_minutes")
    )
    .select(
        "subscriber_id",
        "session_id",
        "session_start",
        "watch_minutes",
        "previous_watch_minutes",
        "watch_minutes_change"
    )
    .orderBy("subscriber_id", "session_start")
)

watch_time_change.show(20, truncate=False)

+-------------+----------+-------------------+-------------+----------------------+--------------------+
|subscriber_id|session_id|session_start      |watch_minutes|previous_watch_minutes|watch_minutes_change|
+-------------+----------+-------------------+-------------+----------------------+--------------------+
|SUB00001     |WS00064390|2026-01-02 17:32:00|58           |NULL                  |NULL                |
|SUB00001     |WS00045183|2026-01-05 04:51:00|78           |58                    |20                  |
|SUB00001     |WS00028484|2026-01-09 08:46:00|4            |78                    |-74                 |
|SUB00001     |WS00043555|2026-01-24 19:41:00|108          |4                     |104                 |
|SUB00001     |WS00041862|2026-02-18 07:43:00|79           |108                   |-29                 |
|SUB00001     |WS00059616|2026-02-25 07:38:00|18           |79                    |-61                 |
|SUB00001     |WS00042345|2026-03-13 07:34:00|44       

**Part E**

In [25]:
from pyspark.sql import functions as F
from pyspark import StorageLevel
from pyspark.sql.window import Window
import time

combined_sessions = sessions_df.unionByName(
    extra_df,
    allowMissingColumns=True
)

before_dedup_count = combined_sessions.count()

deduplicated_sessions = combined_sessions.dropDuplicates(["session_id"])

after_dedup_count = deduplicated_sessions.count()

print("Rows before deduplication:", before_dedup_count)
print("Rows after deduplication:", after_dedup_count)
print("Duplicate rows removed:", before_dedup_count - after_dedup_count)

sessions_df = deduplicated_sessions

Rows before deduplication: 90000
Rows after deduplication: 89400
Duplicate rows removed: 600


In [26]:
from pyspark.sql import functions as F
from pyspark import StorageLevel
from pyspark.sql.window import Window
import time

print([name for name in globals() if "session" in name.lower()])

['SparkSession', 'sessions_df', 'selected_sessions', 'top_20_sessions', 'sessions_with_device_df', 'session_subscribers', 'plan_sessions', 'subscriber_sessions', 'sessions_with_day', 'latest_sessions', 'combined_sessions', 'deduplicated_sessions']


In [28]:
batch1 = sessions_df
batch2 = extra_df

combined_sessions = batch1.unionByName(
    batch2,
    allowMissingColumns=True
)

before_count = combined_sessions.count()

sessions_df = combined_sessions.dropDuplicates(["session_id"])

after_count = sessions_df.count()

print("Rows before deduplication:", before_count)
print("Rows after deduplication:", after_count)
print("Rows removed:", before_count - after_count)

Rows before deduplication: 99400
Rows after deduplication: 89400
Rows removed: 10000


In [29]:
lazy_pipeline = (
    sessions_df
    .filter(F.col("session_status") == "Completed")
    .join(
        content_df.select("content_id", "genre"),
        on="content_id",
        how="left"
    )
    .withColumn(
        "watch_hours",
        F.col("watch_minutes") / 60
    )
    .groupBy("genre")
    .agg(
        F.count("*").alias("session_count"),
        F.sum("watch_hours").alias("total_watch_hours")
    )
    .orderBy(F.desc("total_watch_hours"))
)

lazy_pipeline.explain()
lazy_pipeline.show(truncate=False)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_watch_hours#1501 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_watch_hours#1501 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=3658]
      +- HashAggregate(keys=[genre#100], functions=[count(1), sum(watch_hours#1499)])
         +- Exchange hashpartitioning(genre#100, 200), ENSURE_REQUIREMENTS, [plan_id=3655]
            +- HashAggregate(keys=[genre#100], functions=[partial_count(1), partial_sum(watch_hours#1499)])
               +- Project [genre#100, (cast(watch_minutes#1549 as double) / 60.0) AS watch_hours#1499]
                  +- BroadcastHashJoin [content_id#1543], [content_id#98], LeftOuter, BuildRight, false
                     :- Project [content_id#1543, watch_minutes#1549]
                     :  +- Filter (isnotnull(session_status#1557) AND (session_status#1557 = Completed))
                     :     +- SortAggregate(key=[session_id#17], functions=[first(content_id#1521, false

In [30]:
completed_sessions = (
    sessions_df
    .filter(F.col("session_status") == "Completed")
    .cache()
)

start = time.time()
print("First count:", completed_sessions.count())
print("First count time:", round(time.time() - start, 3), "seconds")

start = time.time()
print("Second count:", completed_sessions.count())
print("Second count time:", round(time.time() - start, 3), "seconds")

start = time.time()
completed_sessions.groupBy("stream_quality").agg(
    F.avg("watch_minutes").alias("avg_watch_minutes")
).show()
print("First aggregation time:", round(time.time() - start, 3), "seconds")

start = time.time()
completed_sessions.groupBy("stream_quality").agg(
    F.avg("watch_minutes").alias("avg_watch_minutes")
).show()
print("Second aggregation time:", round(time.time() - start, 3), "seconds")

completed_sessions.unpersist()

First count: 7112
First count time: 19.013 seconds
Second count: 7112
Second count time: 3.313 seconds
+--------------+-----------------+
|stream_quality|avg_watch_minutes|
+--------------+-----------------+
|       Full HD|77.21663851351352|
|            4K|77.30275229357798|
|            SD|76.22186234817813|
|            HD|76.55306479859895|
+--------------+-----------------+

First aggregation time: 4.505 seconds
+--------------+-----------------+
|stream_quality|avg_watch_minutes|
+--------------+-----------------+
|       Full HD|77.21663851351352|
|            4K|77.30275229357798|
|            SD|76.22186234817813|
|            HD|76.55306479859895|
+--------------+-----------------+

Second aggregation time: 3.777 seconds


DataFrame[session_id: string, subscriber_id: string, content_id: string, device_id: string, session_start: timestamp, watch_minutes: int, completion_pct: double, buffering_seconds: double, stream_quality: string, session_status: string, data_used_mb: double, user_rating: double]

In [31]:
sessions_4k = (
    sessions_df
    .filter(F.col("stream_quality") == "4K")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

print("4K session count:", sessions_4k.count())

sessions_4k.groupBy("session_status").count().show()

sessions_4k.unpersist()

4K session count: 8127
+--------------+-----+
|session_status|count|
+--------------+-----+
|     Completed|  654|
|     Abandoned| 1403|
|   Interrupted| 6070|
+--------------+-----+



DataFrame[session_id: string, subscriber_id: string, content_id: string, device_id: string, session_start: timestamp, watch_minutes: int, completion_pct: double, buffering_seconds: double, stream_quality: string, session_status: string, data_used_mb: double, user_rating: double]

In [33]:
from pyspark.sql.functions import spark_partition_id

sessions_8 = sessions_df.repartition(8)

print("Total partitions:", sessions_8.rdd.getNumPartitions())

partition_counts = (
    sessions_8
    .withColumn("partition_id", spark_partition_id())
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")
)

partition_counts.show()

Total partitions: 8
+------------+-----+
|partition_id|count|
+------------+-----+
|           0|11175|
|           1|11175|
|           2|11175|
|           3|11175|
|           4|11175|
|           5|11175|
|           6|11175|
|           7|11175|
+------------+-----+



In [34]:
sessions_8 = sessions_df.repartition(8)

sessions_coalesce_2 = sessions_8.coalesce(2)
sessions_repartition_2 = sessions_8.repartition(2)

print("Original partitions:", sessions_8.rdd.getNumPartitions())
print("Coalesce partitions:", sessions_coalesce_2.rdd.getNumPartitions())
print("Repartition partitions:", sessions_repartition_2.rdd.getNumPartitions())

print("Coalesce execution plan:")
sessions_coalesce_2.explain()

print("Repartition execution plan:")
sessions_repartition_2.explain()

Original partitions: 8
Coalesce partitions: 2
Repartition partitions: 2
Coalesce execution plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 3
   +- Coalesce 2
      +- ShuffleQueryStage 2
         +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=6004]
            +- SortAggregate(key=[session_id#17], functions=[first(subscriber_id#5608, false), first(content_id#5610, false), first(device_id#5612, false), first(session_start#5614, false), first(watch_minutes#5616, false), first(completion_pct#5618, false), first(buffering_seconds#5620, false), first(stream_quality#5622, false), first(session_status#5624, false), first(data_used_mb#5626, false), first(user_rating#5628, false)])
               +- *(5) Sort [session_id#17 ASC NULLS FIRST], false, 0
                  +- AQEShuffleRead coalesced
                     +- ShuffleQueryStage 1
                        +- Exchange hashpartitioning(session_id#17, 200), ENSURE_REQ

**Final Capstone**

In [35]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time

combined_sessions = sessions_df.unionByName(
    extra_df,
    allowMissingColumns=True
)

before_count = combined_sessions.count()

sessions_clean = combined_sessions.dropDuplicates(["session_id"])

after_count = sessions_clean.count()

print("Rows before deduplication:", before_count)
print("Rows after deduplication:", after_count)
print("Duplicates removed:", before_count - after_count)

Rows before deduplication: 99400
Rows after deduplication: 89400
Duplicates removed: 10000


In [36]:
sessions_clean = (
    sessions_clean
    .filter(F.col("session_id").isNotNull())
    .filter(F.col("session_start").isNotNull())
    .filter(F.col("content_id").isNotNull())
    .withColumn("session_start", F.to_timestamp("session_start"))
    .withColumn("watch_minutes", F.col("watch_minutes").cast("double"))
    .withColumn("completion_pct", F.col("completion_pct").cast("double"))
    .withColumn("buffering_seconds", F.col("buffering_seconds").cast("double"))
    .withColumn("data_used_mb", F.col("data_used_mb").cast("double"))
)

print("Cleaned session count:", sessions_clean.count())
sessions_clean.show(5, truncate=False)

Cleaned session count: 89400
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00000002|SUB01430     |CT0031    |DV04     |2026-07-30 06:26:00|31.0         |56.4          |31.0             |HD            |Interrupted   |422.7       |NULL       |
|WS00000004|SUB02017     |CT0040    |DV08     |2026-02-20 17:07:00|69.0         |75.0          |2.0              |HD            |Interrupted   |924.7       |NULL       |
|WS00000005|SUB01268     |CT0052    |DV08     |2026-04-06 04:33:00|36.0         |23.4          |31.0             |4K     

In [37]:
report_base = (
    sessions_clean
    .join(
        subscribers_df.select("subscriber_id", "region", "plan_id"),
        on="subscriber_id",
        how="left"
    )
    .join(
        content_df.select("content_id", "content_title", "genre"),
        on="content_id",
        how="left"
    )
    .join(
        devices_df.select("device_id", "device_type"),
        on="device_id",
        how="left"
    )
    .join(
        plans_df.select("plan_id", "plan_name"),
        on="plan_id",
        how="left"
    )
    .withColumn(
        "month",
        F.date_format("session_start", "yyyy-MM")
    )
)

report_base.select(
    "session_id",
    "month",
    "region",
    "genre",
    "content_title",
    "device_type",
    "plan_name"
).show(10, truncate=False)

+----------+-------+------+-----------+---------------+--------------+---------+
|session_id|month  |region|genre      |content_title  |device_type   |plan_name|
+----------+-------+------+-----------+---------------+--------------+---------+
|WS00000002|2026-07|West  |Animation  |Golden Archive |Tablet        |Basic    |
|WS00000004|2026-02|West  |Adventure  |Emerald Echo   |Laptop Browser|Mobile   |
|WS00000005|2026-04|South |Sports     |Midnight Echo  |Laptop Browser|Standard |
|WS00000006|2026-03|West  |Adventure  |Fading Harbor  |Smart TV      |Basic    |
|WS00000008|2026-08|East  |Thriller   |Silver Puzzle  |Laptop Browser|Premium  |
|WS00000011|2026-03|West  |Comedy     |Ancient Valley |iPhone        |Mobile   |
|WS00000014|2026-09|South |Comedy     |Ancient Circuit|iPhone        |Basic    |
|WS00000016|2026-08|North |Animation  |Hidden Circuit |Smart TV      |Mobile   |
|WS00000021|2026-08|East  |Documentary|Quiet Frontier |Android Phone |Standard |
|WS00000029|2026-01|South |A

In [38]:
monthly_report = (
    report_base
    .groupBy("month", "region", "genre")
    .agg(
        F.count("*").alias("session_count"),
        (F.sum("watch_minutes") / 60).alias("watch_hours"),
        F.avg("completion_pct").alias("completion_rate"),
        F.avg("buffering_seconds").alias("average_buffering_seconds"),
        F.sum("data_used_mb").alias("total_data_usage_mb")
    )
    .orderBy("month", "region", "genre")
)

monthly_report.show(50, truncate=False)

+-------+-------+-----------+-------------+------------------+------------------+-------------------------+-------------------+
|month  |region |genre      |session_count|watch_hours       |completion_rate   |average_buffering_seconds|total_data_usage_mb|
+-------+-------+-----------+-------------+------------------+------------------+-------------------------+-------------------+
|2026-01|NULL   |Adventure  |37           |38.35             |61.8742857142857  |37.42424242424242        |44063.59999999999  |
|2026-01|NULL   |Animation  |21           |13.016666666666667|54.75238095238095 |29.68421052631579        |13333.199999999999 |
|2026-01|NULL   |Comedy     |9            |4.416666666666667 |36.588888888888896|30.77777777777778        |5805.3             |
|2026-01|NULL   |Documentary|22           |11.666666666666666|51.56818181818182 |40.86363636363637        |13426.2            |
|2026-01|NULL   |Drama      |23           |17.216666666666665|54.31904761904761 |34.09090909090909      

In [39]:
title_metrics = (
    report_base
    .filter(F.col("content_title").isNotNull())
    .groupBy("genre", "content_id", "content_title")
    .agg(
        (F.sum("watch_minutes") / 60).alias("watch_hours")
    )
)

title_window = Window.partitionBy("genre").orderBy(
    F.desc("watch_hours")
)

top_three_titles = (
    title_metrics
    .withColumn("rank", F.dense_rank().over(title_window))
    .filter(F.col("rank") <= 3)
    .orderBy("genre", "rank")
)

top_three_titles.show(50, truncate=False)

+-----------+----------+----------------+------------------+----+
|genre      |content_id|content_title   |watch_hours       |rank|
+-----------+----------+----------------+------------------+----+
|Adventure  |CT0105    |Infinite Horizon|1210.8            |1   |
|Adventure  |CT0042    |Solar Memory    |1129.4666666666667|2   |
|Adventure  |CT0048    |Lost Journey    |1127.85           |3   |
|Animation  |CT0106    |Last River      |969.3             |1   |
|Animation  |CT0114    |Emerald Circuit |963.0833333333334 |2   |
|Animation  |CT0100    |Golden Puzzle   |932.2166666666667 |3   |
|Comedy     |CT0019    |Bright Valley   |1173.8            |1   |
|Comedy     |CT0027    |Blue Horizon    |1151.25           |2   |
|Comedy     |CT0056    |Wild Canvas     |1136.2333333333333|3   |
|Documentary|CT0077    |Fading Signal   |878.6833333333333 |1   |
|Documentary|CT0057    |Emerald Valley  |743.8833333333333 |2   |
|Documentary|CT0014    |Solar Horizon   |684.2666666666667 |3   |
|Drama    

In [40]:
final_report = monthly_report.cache()

start = time.time()
first_count = final_report.count()
first_time = time.time() - start

print("First action — row count:", first_count)
print("First action time:", round(first_time, 3), "seconds")

start = time.time()
final_report.show(20, truncate=False)
second_time = time.time() - start

print("Second action time:", round(second_time, 3), "seconds")

final_report.unpersist()

print("Final report cache released.")

First action — row count: 432
First action time: 12.138 seconds
+-------+-------+-----------+-------------+------------------+------------------+-------------------------+-------------------+
|month  |region |genre      |session_count|watch_hours       |completion_rate   |average_buffering_seconds|total_data_usage_mb|
+-------+-------+-----------+-------------+------------------+------------------+-------------------------+-------------------+
|2026-01|NULL   |Adventure  |37           |38.35             |61.8742857142857  |37.42424242424242        |44063.59999999999  |
|2026-01|NULL   |Animation  |21           |13.016666666666667|54.75238095238095 |29.68421052631579        |13333.199999999999 |
|2026-01|NULL   |Comedy     |9            |4.416666666666667 |36.588888888888896|30.77777777777778        |5805.3             |
|2026-01|NULL   |Documentary|22           |11.666666666666666|51.56818181818182 |40.86363636363637        |13426.2            |
|2026-01|NULL   |Drama      |23         

In [41]:
output_path = "/content/final_streaming_performance_report"

(
    final_report
    .repartition(3)
    .write
    .mode("overwrite")
    .parquet(output_path)
)

print("Report saved to:", output_path)

Report saved to: /content/final_streaming_performance_report


In [42]:
import os

saved_report = spark.read.parquet(output_path)

print("Saved report row count:", saved_report.count())

print("Saved report schema:")
saved_report.printSchema()

print("Saved report preview:")
saved_report.show(20, truncate=False)

print("Output files:")
for filename in sorted(os.listdir(output_path)):
    print(filename)

Saved report row count: 432
Saved report schema:
root
 |-- month: string (nullable = true)
 |-- region: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- session_count: long (nullable = true)
 |-- watch_hours: double (nullable = true)
 |-- completion_rate: double (nullable = true)
 |-- average_buffering_seconds: double (nullable = true)
 |-- total_data_usage_mb: double (nullable = true)

Saved report preview:
+-------+-------+-----------+-------------+------------------+------------------+-------------------------+-------------------+
|month  |region |genre      |session_count|watch_hours       |completion_rate   |average_buffering_seconds|total_data_usage_mb|
+-------+-------+-----------+-------------+------------------+------------------+-------------------------+-------------------+
|2026-03|East   |Comedy     |152          |140.25            |59.02972972972973 |33.22222222222222        |149988.2           |
|2026-08|West   |Thriller   |184          |119.95         